# Resume Phase 3 from the verified eight-response checkpoint

**First choose File → Save a copy in Drive, and work in that copy. Keep cell outputs included when saving.** Then select a **T4 GPU** runtime. This notebook needs no API key, Drive mount or payment details. If GPU quota is unavailable, stop here; this notebook does not purchase access.

The pinned checkout contains the verified 04:28:37 UTC checkpoint and `docs/PHASE3_RESUME_PROTOCOL.md`. It preserves eight responses and the original vectors, then fills the 190 missing grid positions. Earlier unarchived attempts are unknown: **198 retained outcomes would not establish exactly 198 total attempts**. Invalid/truncated retained responses are never retried.

Run cells in order. Setup refuses to overwrite an existing output directory. Keep the foreground collection cell running. It updates one compact JSON snapshot output after at most ten new rows and on interruption/completion. A displayed snapshot is **not durable until its full output is actually saved to Drive or exported outside the VM**. Verify that the saved notebook contains the latest complete `phase3-recovery-snapshot-v1` JSON output; autosave must not be assumed. Save periodically with Ctrl/Cmd+S. Do not rerun the collection cell after it has started; preserve its latest snapshot before handling another interruption.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, re
from datetime import datetime, timezone

COPIED_TO_DRIVE = False  # Set True only in the saved Drive copy, with outputs included.
CHECKPOINT_COMMIT = "3cf3f5b357ce273540d98dfcf2df1e64c96bfa6e"
SCIENTIFIC_COMMIT = "ef90362c287774a9d22ffa05a87f9faa960cad4e"
if COPIED_TO_DRIVE is not True:
    raise RuntimeError("Save a copy in Drive with outputs included, then set COPIED_TO_DRIVE=True in that copy.")
if not re.fullmatch(r"[0-9a-f]{40}", CHECKPOINT_COMMIT):
    raise ValueError("CHECKPOINT_COMMIT must be the published full commit hash.")
PROJECT = Path('/content') / ('hint-faithfulness-phase3-resume-' + CHECKPOINT_COMMIT[:12])
if PROJECT.is_symlink():
    raise RuntimeError("The dedicated checkout must not be a symlink.")
if not PROJECT.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/felmonon/hint-faithfulness.git',str(PROJECT)],check=True)
    subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',CHECKPOINT_COMMIT],check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
changes = subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
if head != CHECKPOINT_COMMIT or changes:
    raise RuntimeError("Existing checkout differs; preserve it and use a fresh dedicated checkout.")
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
os.environ['CUDA_VISIBLE_DEVICES']='0'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-pilot.txt'],check=True)
print('Pinned checkpoint checkout:',head)
print('No generation has started.')

In [ ]:
import shutil
import numpy as np
import torch, transformers
from collections import Counter
from src.common import LETTERS, read_jsonl, stable_hash
from src.phase3 import load_records, paired_prompts, target_stratified_split
from src.provenance import capture

def stamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()
def sha_file(path):
    return sha_bytes(Path(path).read_bytes())
def require(condition, message):
    if not condition:
        raise RuntimeError(message)

require(torch.cuda.is_available(),'A CUDA T4 is required. No checkpoint has been restored or generation started.')
gpu=torch.cuda.get_device_properties(0)
require('T4' in gpu.name and gpu.total_memory >= 14*1024**3,'This recovery requires a T4 with at least 14 GiB.')
require(np.__version__=='2.2.6' and transformers.__version__=='5.0.0rc3',
        'Imported NumPy/Transformers differ from pinned requirements; resolve the environment before restoring outputs.')
CHECKPOINT=PROJECT/'artifacts/pilot-20-phase3-checkpoint-0430'
PHASE3=PROJECT/'artifacts/pilot-20-phase3-v1'
require(not PHASE3.exists(),'Output already exists. Preserve it; this notebook never overwrites or restarts it.')
EXPECTED_PAYLOAD_HASHES={
 'manifest.json':'b6ab6444ac32113f1e00ebf3f1cba43a6344f00ada0b431ff7b4fdc03b74565c',
 'plan.json':'5ecc12c4f4d79161de442582f2bea7b89815a0ba06808eb130b00054de52d1f2',
 'runtime.json':'46c1201b9755f0ac8f317773cfdff9e82ed5c5ae966ee3528ff88232e59d0b64',
 'steering.jsonl':'5bec0702c11667104b6a37a3f15487b19dceb7358f771748e467b346ae63044b',
 'vector_metadata.json':'cf3a7ff327d4ef6eb496a0819c8af2539b1d5ee43636db284cbb5266dc0cba20',
 'vectors.npz':'9808b154bd3e584c53ba8fe784d3b5d47c67ec131a9c246fe0ca5dfc828e8c57',
 'execution.log':'5b580df84a568b4de5ad1f601844354302fdfaaf01e351f31d1e417d015373d9'}
EXPECTED_CODE_HASHES={
 'src/phase3.py':'57bb529ac9096f8892d3da6253d43b728d7fb562b1c1565a64c5b0691a28fcdc',
 'src/local_model.py':'b7a549ace225611660eb0c27ca03ba12cb1d4a180407b7138c1fd223fdb68766',
 'src/common.py':'665d1d7866e87dac4afd0cff9e12c7be190a57e6daaa9cb692441e1475cc9d9d',
 'scripts/run_phase3_pilot.py':'8e616c5b20f21c01682e215ea2ba0e8b5bf0ed8d145fd529c6497190474ef714'}
for name,expected in EXPECTED_PAYLOAD_HASHES.items():
    require(not (CHECKPOINT/name).is_symlink() and sha_file(CHECKPOINT/name)==expected,'Checkpoint mismatch: '+name)
for name,expected in EXPECTED_CODE_HASHES.items():
    require(sha_file(PROJECT/name)==expected,'Frozen scientific source differs: '+name)
provenance=json.loads((CHECKPOINT/'checkpoint-provenance.json').read_text())
require(provenance['files_sha256']==EXPECTED_PAYLOAD_HASHES,'Checkpoint acquisition hashes differ.')
PLAN=json.loads((CHECKPOINT/'plan.json').read_text())
require(PLAN==json.loads((PROJECT/'docs/phase3-pilot-plan.json').read_text()),'Frozen plan differs.')
source_path=PROJECT/PLAN['source_path']
require(sha_file(source_path)==PLAN['source_sha256'],'Frozen Phase 1 source differs.')
source,identity=load_records(source_path,include_invalid_source_outcomes=True)
baselines,hinted,qids=paired_prompts(source,require_complete=True)
targets={q:hinted[q][0]['hint_letter'] for q in qids}
train,test=target_stratified_split(qids,targets,PLAN['seed'])
require(train==PLAN['train_question_ids'] and test==PLAN['test_question_ids'],'Split differs.')
MANIFEST=json.loads((CHECKPOINT/'manifest.json').read_text())
expected_manifest={**identity,'source_digest':stable_hash(source),'layer':PLAN['layer'],
 'position':'post_decoder_block_residual_before_final_norm','seed':PLAN['seed'],
 'strengths':[float(s) for s in PLAN['strengths']],'max_new_tokens':PLAN['max_new_tokens'],
 'temperature':PLAN['temperature'],'top_p':PLAN['top_p'],'top_k':PLAN['top_k'],
 'source_dtypes':sorted({r.get('dtype','unknown') for r in source}),'intervention_dtype':PLAN['dtype'],
 'source_prompt_tokens_verified':True,'activation_token_position':'last_original_chat_prompt_token',
 'split_rule':PLAN['split_rule'],'assigned_targets':targets,'training_question_ids':train,'test_question_ids':test}
require(MANIFEST==expected_manifest,'Checkpoint manifest differs from frozen source/plan.')
meta=json.loads((CHECKPOINT/'vector_metadata.json').read_text())
require(meta['training_questions_per_letter']==dict(Counter(targets[q] for q in train))==PLAN['train_target_counts'],
        'Vector training counts differ.')
rng=np.random.default_rng(PLAN['seed'])
with np.load(CHECKPOINT/'vectors.npz',allow_pickle=False) as vectors:
    require(set(vectors.files)=={f'{d}_{l}' for d in ('vector','random') for l in LETTERS},'Vector keys differ.')
    for letter in LETTERS:
        v,r=vectors[f'vector_{letter}'],vectors[f'random_{letter}']
        require(v.shape==r.shape==(2048,) and v.dtype==r.dtype==np.dtype('float32'),'Vector shape/type differs.')
        require(np.isfinite(v).all() and np.isfinite(r).all(),'Nonfinite vector.')
        norm=float(np.linalg.norm(v))
        require(np.isclose(norm,meta['vector_norms'][letter],rtol=2e-7,atol=0) and norm>0,'Vector norm differs.')
        z=rng.standard_normal(v.shape).astype('float32')
        require(np.allclose(r,z*norm/np.linalg.norm(z),rtol=2e-7,atol=1e-7),'Seeded random control differs.')
ORIGINAL_PREFIX=(CHECKPOINT/'steering.jsonl').read_bytes()
require(ORIGINAL_PREFIX.endswith(b'\n'),'Checkpoint must end on a complete JSONL boundary.')
conditions=[('baseline',None,0.0)]+[(d,l,float(s)) for d in ('vector','random') for l in LETTERS for s in PLAN['strengths'] if s>0]
EXPECTED_IDS=[stable_hash([stable_hash(MANIFEST),q,d,l,s]) for q in test for d,l,s in conditions]
require(len(EXPECTED_IDS)==198 and len(set(EXPECTED_IDS))==198,'Unexpected condition grid.')
original_rows=[json.loads(line) for line in ORIGINAL_PREFIX.splitlines()]
require(len(original_rows)==8 and [r['record_id'] for r in original_rows]==EXPECTED_IDS[:8],'Checkpoint is not the eight-row grid prefix.')
# All validation above precedes restoration. Never replace existing output files.
PHASE3.mkdir(parents=True,exist_ok=False)
for name in EXPECTED_PAYLOAD_HASHES:
    with (PHASE3/name).open('xb') as output:
        output.write((CHECKPOINT/name).read_bytes())
METADATA=PROJECT/'artifacts'/('pilot-20-phase3-resume-notebook-'+stamp())
METADATA.mkdir(parents=True,exist_ok=False)
RESUME_LOG=METADATA/'resume-execution.log'  # Outside the runner's checksummed artifact directory.
SETUP_PATH=METADATA/'setup.json'
new_runtime=capture();old_runtime=json.loads((CHECKPOINT/'runtime.json').read_text())
runtime_changes={key:{'original':old_runtime.get(key),'resumed':new_runtime.get(key)}
                 for key in ('python','torch_version','cuda_version','gpu_name','nvidia_driver')
                 if old_runtime.get(key)!=new_runtime.get(key)}
SETUP_PATH.write_text(json.dumps({'checkpoint_commit':CHECKPOINT_COMMIT,'scientific_commit':SCIENTIFIC_COMMIT,
 'runtime':new_runtime,'imported_numpy':np.__version__,'imported_transformers':transformers.__version__,
 'runtime_changes':runtime_changes,'retained_original_records':8,'planned_missing_positions':190,
 'historical_total_attempts':'unknown; unarchived original attempts may exist'},indent=2)+'\n')
print('Restored eight responses; original execution.log/runtime.json preserved; vectors will be reused.')
print('GPU:',gpu.name,'| imported Torch:',torch.__version__,'| recorded runtime differences:',runtime_changes)
print('No generation has started. Read the recovery protocol in the pinned checkout.')

## Checkpoint output and foreground collection

The next helper cell defines the snapshot format; the following cell starts the **190-position continuation** and remains busy until it ends. Full subprocess output streams to a separate `resume-execution.log`; only concise progress appears below. Interrupting the cell terminates its process group and attempts a last snapshot.

Each snapshot contains only named experiment files. Each file has `encoding: gzip+base64`, compressed and original SHA-256 values, original byte count, and base64 data. Decompressing restores the exact original bytes. `steering.jsonl` is always cut at a completed newline and must retain the original eight-row byte prefix. The saved snapshot includes original checkpoint evidence, current vectors/manifest/plan, original and resumed runtime records, and the separate resume log. This avoids a growing log racing the runner's final checksum inventory.

**Save the Drive notebook with outputs and verify the latest snapshot actually persisted.** A display update or a local file alone is not an external backup. Do not claim durable recovery from an unsaved output. The final cell creates both a ZIP and standalone snapshot JSON; if downloading fails, the complete saved JSON output is the fallback.

In [ ]:
import base64, gzip
from IPython.display import JSON, display

FINAL_FILES={'summary.jsonl','explanation_labels.csv','EXPLANATION_SUMMARY.md','LABELING.md','steering_rates.png','completion.json','checksums.sha256'}
SNAPSHOT_DISPLAY=None
LATEST_SNAPSHOT=None
RUN_STARTED=False
RUN_COMPLETED=False

def completed_jsonl(raw):
    boundary=raw.rfind(b'\n')+1
    prefix=raw[:boundary]
    rows=[json.loads(line) for line in prefix.splitlines()]
    return prefix,rows

def current_prefix(max_records=None):
    raw=(PHASE3/'steering.jsonl').read_bytes()
    prefix,rows=completed_jsonl(raw)
    require(prefix.startswith(ORIGINAL_PREFIX),'Original eight-row byte prefix changed.')
    require(8<=len(rows)<=198 and [r['record_id'] for r in rows]==EXPECTED_IDS[:len(rows)],
            'Duplicate, missing, reordered or unexpected grid IDs.')
    if max_records is not None:
        require(8<=max_records<=len(rows),'Requested snapshot prefix is unavailable.')
        prefix=b''.join(prefix.splitlines(keepends=True)[:max_records]);rows=rows[:max_records]
    return prefix,rows

def pack_bytes(raw):
    compressed=gzip.compress(raw,mtime=0)
    return {'encoding':'gzip+base64','bytes':len(raw),'sha256':sha_bytes(raw),
            'compressed_bytes':len(compressed),'compressed_sha256':sha_bytes(compressed),
            'base64':base64.b64encode(compressed).decode('ascii')}

def unpack_bytes(item):
    require(item['encoding']=='gzip+base64','Unknown snapshot encoding.')
    compressed=base64.b64decode(item['base64'],validate=True)
    require(len(compressed)==item['compressed_bytes'] and sha_bytes(compressed)==item['compressed_sha256'],
            'Compressed snapshot payload differs.')
    raw=gzip.decompress(compressed)
    require(len(raw)==item['bytes'] and sha_bytes(raw)==item['sha256'],'Decoded snapshot payload differs.')
    return raw

def snapshot(complete=False,max_records=None):
    global SNAPSHOT_DISPLAY,LATEST_SNAPSHOT
    prefix,rows=current_prefix(max_records)
    for name,expected in EXPECTED_PAYLOAD_HASHES.items():
        if name!='steering.jsonl':
            require(sha_file(PHASE3/name)==expected,'Restored immutable payload changed: '+name)
    paths=[PHASE3/name for name in EXPECTED_PAYLOAD_HASHES if name!='steering.jsonl']
    paths += sorted(PHASE3.glob('runtime-resume-*.json'))
    paths += [CHECKPOINT/name for name in EXPECTED_PAYLOAD_HASHES]
    paths += [CHECKPOINT/'checkpoint-provenance.json',SETUP_PATH]
    if RESUME_LOG.exists():paths.append(RESUME_LOG)
    if complete:
        require(len(rows)==198,'Cannot mark an incomplete snapshot complete.')
        paths += [PHASE3/name for name in sorted(FINAL_FILES) if (PHASE3/name).exists()]
    data={str((PHASE3/'steering.jsonl').relative_to(PROJECT)):pack_bytes(prefix)}
    for path in paths:
        require(path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(PROJECT.resolve()),
                'Unexpected snapshot file.')
        data[str(path.relative_to(PROJECT))]=pack_bytes(path.read_bytes())
    if complete:
        inventory=[line.split('  ',1)[1] for line in (PHASE3/'checksums.sha256').read_text().splitlines()]
        require(all(str((PHASE3/name).relative_to(PROJECT)) in data for name in inventory),
                'A checksummed artifact is missing from the complete snapshot.')
    payload={'schema':'phase3-recovery-snapshot-v1','captured_at_utc':datetime.now(timezone.utc).isoformat(),
        'checkpoint_commit':CHECKPOINT_COMMIT,'scientific_commit':SCIENTIFIC_COMMIT,
        'retained_records':len(rows),'original_retained_records':8,'complete_verified':complete,
        'historical_total_attempts':'unknown; may exceed retained grid outcomes',
        'durability':'Unverified until this complete output or exported file is saved outside the runtime.',
        'files':data}
    LATEST_SNAPSHOT=payload
    if SNAPSHOT_DISPLAY is None:
        SNAPSHOT_DISPLAY=display(JSON(payload,expanded=False),display_id=True)
    else:
        SNAPSHOT_DISPLAY.update(JSON(payload,expanded=False))
    print(f'Checkpoint output updated: {len(rows)}/198 retained records. Save and verify the Drive copy.',flush=True)
    return payload

def verify_completion():
    prefix,rows=current_prefix()
    require((PHASE3/'steering.jsonl').read_bytes()==prefix,'Trailing incomplete JSONL bytes remain.')
    require(len(rows)==198,'The 198-position grid is incomplete.')
    completion=json.loads((PHASE3/'completion.json').read_text())
    require(completion.get('status')=='complete_fixed_sweep' and completion.get('records')==198,'Completion marker differs.')
    for name,expected in EXPECTED_PAYLOAD_HASHES.items():
        if name!='steering.jsonl':require(sha_file(PHASE3/name)==expected,'Immutable payload changed: '+name)
    names=set()
    for line in (PHASE3/'checksums.sha256').read_text().splitlines():
        expected,name=line.split('  ',1);path=PHASE3/name
        require(name not in names and name!='checksums.sha256','Duplicate or self-referential checksum.')
        require(not path.is_symlink() and path.resolve().is_relative_to(PHASE3.resolve()),'Unsafe checksum path.')
        require(sha_file(path)==expected,'Completed artifact checksum differs: '+name)
        names.add(name)
    require(set(EXPECTED_PAYLOAD_HASHES)|{'completion.json','summary.jsonl','explanation_labels.csv','EXPLANATION_SUMMARY.md','LABELING.md','steering_rates.png'}<=names,
            'Incomplete artifact checksum inventory.')
    require(bool(list(PHASE3.glob('runtime-resume-*.json'))),'No separate resumed-runtime provenance.')
    from scripts.analyze_phase3_pilot import validate_run
    validate_run(rows,MANIFEST,PLAN,source)
    return len(rows)

print('Snapshot helpers ready. No generation started.')

In [ ]:
import queue, signal, threading, time
require(RUN_STARTED is False,'This collection cell already started; preserve its output instead of rerunning it.')
require(COPIED_TO_DRIVE is True,'Use a saved Drive copy with outputs included.')
require(not RESUME_LOG.exists(),'Resume log already exists; preserve the prior run.')
require(len(current_prefix()[1])==8,'This notebook starts only from the verified eight-response checkpoint.')
RUN_STARTED=True
command=[sys.executable,'-u','scripts/run_phase3_pilot.py','--resume']
events=queue.Queue();reader_errors=[]
process=None;reader=None
last_snapshot_n=8
snapshot()  # Recoverable initial state, still requiring an external save.
log=RESUME_LOG.open('xb')
try:
    process=subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             start_new_session=True,bufsize=0)
    def copy_output():
        try:
            for raw in iter(process.stdout.readline,b''):
                log.write(raw);log.flush()
                text=raw.decode('utf-8',errors='replace')
                if 'Saved steering response ' in text or 'PHASE3_' in text:
                    events.put(text.rstrip())
        except BaseException as exc:
            reader_errors.append(exc)
    reader=threading.Thread(target=copy_output,daemon=True);reader.start()
    while process.poll() is None or reader.is_alive():
        if process.poll() is not None and process.returncode!=0:
            raise RuntimeError('Runner exited unsuccessfully; stopping any surviving child and preserving a snapshot.')
        while not events.empty():print(events.get_nowait(),flush=True)
        if reader_errors:raise RuntimeError('Log reader failed; preserve partial evidence.') from reader_errors[0]
        count=len(current_prefix()[1])
        while count-last_snapshot_n>=10:
            last_snapshot_n+=10;snapshot(max_records=last_snapshot_n)
        time.sleep(.5)
    reader.join()
    while not events.empty():print(events.get_nowait(),flush=True)
    if reader_errors:raise RuntimeError('Log reader failed.') from reader_errors[0]
    require(process.wait()==0,'Runner stopped. Preserve the latest snapshot and resume log; do not rerun retained outputs.')
    log.flush()
    verify_completion()
    RUN_COMPLETED=True
    snapshot(complete=True)
    print('Verified 198 retained grid outcomes. Historical total attempts remain unknown. Run the export cell.')
except BaseException:
    try:
        if process is not None:
            # Also stop descendants if their leader has already exited.
            try:os.killpg(process.pid,signal.SIGTERM)
            except ProcessLookupError:pass
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError:pass
                process.wait(timeout=10)
        if reader is not None:
            reader.join(timeout=10)
            if reader.is_alive() and process is not None:
                try:os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError:pass
                reader.join(timeout=10)
    except Exception as cleanup_error:
        print('Process cleanup warning:',type(cleanup_error).__name__)
    finally:
        try:log.flush()
        except Exception as flush_error:print('Log flush warning:',type(flush_error).__name__)
        try:snapshot(complete=False)
        except Exception as backup_error:print('Final snapshot failed:',type(backup_error).__name__,'Preserve the prior saved snapshot.')
    raise
finally:
    log.close()


In [ ]:
import zipfile
from google.colab import files
require(RUN_COMPLETED is True,'The foreground run has not completed; preserve the partial JSON snapshot instead.')
verify_completion()
payload=snapshot(complete=True)
export_stamp=stamp()
archive_path=Path('/content')/f'pilot-20-phase3-resumed-{export_stamp}.zip'
snapshot_path=Path('/content')/f'pilot-20-phase3-snapshot-{export_stamp}.json'
with snapshot_path.open('x',encoding='utf-8') as stream:
    json.dump(payload,stream,separators=(',',':'));stream.write('\n')
with zipfile.ZipFile(archive_path,'x',zipfile.ZIP_DEFLATED) as archive:
    for relative,item in sorted(payload['files'].items()):
        rel=Path(relative)
        require(not rel.is_absolute() and '..' not in rel.parts,'Unsafe export path.')
        archive.writestr(relative,unpack_bytes(item))
    archive.writestr('export-manifest.json',json.dumps({
        'schema':payload['schema'],'checkpoint_commit':CHECKPOINT_COMMIT,'retained_records':198,
        'historical_total_attempts':payload['historical_total_attempts'],
        'files':{name:{'bytes':item['bytes'],'sha256':item['sha256']} for name,item in payload['files'].items()}},indent=2)+'\n')
print('ZIP:',archive_path.name,'bytes:',archive_path.stat().st_size,'SHA256:',sha_file(archive_path))
print('Fallback JSON:',snapshot_path.name,'bytes:',snapshot_path.stat().st_size,'SHA256:',sha_file(snapshot_path))
print('The complete structured JSON snapshot is also in the foreground cell output. Save and verify the Drive copy.')
try:
    files.download(str(archive_path))
except Exception as error:
    print('Browser download did not complete:',type(error).__name__)
    print('Use the saved full JSON output or download the standalone JSON from the Files panel.')

## Restoring a saved JSON snapshot

The schema is `phase3-recovery-snapshot-v1`. For **every** `files[relative_path]` entry: base64-decode `base64`; verify `compressed_bytes` and `compressed_sha256`; gzip-decompress; verify `bytes` and `sha256`; then write the bytes to a **new**, empty recovery directory using that relative path. Reject absolute paths, `..` and symlinks. The helper `unpack_bytes` implements the byte/hash checks. Verify every entry before using the snapshot. A truncated JSON object, missing file or failed hash is not a recoverable checkpoint.

A notebook output saved through `display_id.update` must be checked in the exported/saved notebook itself. This notebook cannot inspect Drive autosave and therefore never asserts external durability. Keep the original checkpoint and each verified acquired snapshot. An updated on-screen count alone is insufficient evidence of a backup.

Do not use this notebook to restart from eight after a later interruption. Preserve the latest complete saved snapshot, document the incident, and restore that newer prefix for a deliberate identical-manifest continuation. Never silently refit vectors, replace a retained failed response, or choose outputs by quality.

After a completed export, perform the full artifact audit and the frozen paired/attrition analysis. The notebook creates no explanation labels and makes no steering-success or faithfulness claim. Human validation remains absent.